In [1]:
import sqlite3
import pandas as pd
import numpy as np

print("Loading data and engineering Momentum features...")

# 1. Load the baseline features from your massive database
conn = sqlite3.connect('../data/omni_pundit.db')
df = pd.read_sql("SELECT * FROM baseline_features ORDER BY date ASC", conn)
conn.close()

# Convert date strings to actual datetime objects
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Recreate the canonicalized player keys directly in Pandas, so we don't have to alter the SQLite database.
df['white_key'] = df['white'].str.replace(" ", "").str.strip().str.lower()
df['black_key'] = df['black'].str.replace(" ", "").str.strip().str.lower()

# 2. Reshape the data: Create a timeline for every individual player
player_games = []

# (Using df.itertuples() is about 10x faster than iterrows() for large datasets!)
for row in df.itertuples():
    # Record the match from White's perspective
    player_games.append({
        'game_id': row.game_id,
        'date': row.date,
        'player': row.white_key,
        'color': 'white',
        'elo': row.white_elo,
        'score': 1.0 if row.target_result == 1 else (0.5 if row.target_result == 0 else 0.0)
    })
    # Record the match from Black's perspective
    player_games.append({
        'game_id': row.game_id,
        'date': row.date,
        'player': row.black_key,
        'color': 'black',
        'elo': row.black_elo,
        'score': 1.0 if row.target_result == -1 else (0.5 if row.target_result == 0 else 0.0)
    })

pdf = pd.DataFrame(player_games).sort_values('date')

# 3. Calculate Rolling Features (WITH LEAKAGE PROTECTION)
WINDOW = 10

# CRITICAL: We .shift(1) so the result of the CURRENT game isn't included in the past form!
pdf['past_score'] = pdf.groupby('player')['score'].shift(1)
pdf['past_elo'] = pdf.groupby('player')['elo'].shift(1)

# Calculate win rate over the last 10 games
pdf['rolling_score'] = pdf.groupby('player')['past_score'].rolling(window=WINDOW, min_periods=1).mean().reset_index(0, drop=True)

# Calculate Elo trend (Current Elo minus Elo 10 games ago)
pdf['elo_10_games_ago'] = pdf.groupby('player')['past_elo'].shift(WINDOW - 1)
pdf['elo_trend'] = pdf['elo'] - pdf['elo_10_games_ago']

# 4. Merge the engineered features back into the main DataFrame
white_features = pdf[pdf['color'] == 'white'][['game_id', 'rolling_score', 'elo_trend']].rename(
    columns={'rolling_score': 'w_form_10', 'elo_trend': 'w_elo_trend'}
)
black_features = pdf[pdf['color'] == 'black'][['game_id', 'rolling_score', 'elo_trend']].rename(
    columns={'rolling_score': 'b_form_10', 'elo_trend': 'b_elo_trend'}
)

df = df.merge(white_features, on='game_id', how='left')
df = df.merge(black_features, on='game_id', how='left')

# Fill missing values (for a player's first few games in the database)
df.fillna({'w_form_10': 0.5, 'b_form_10': 0.5, 'w_elo_trend': 0, 'b_elo_trend': 0}, inplace=True)

# Create the final differentials that the Machine Learning model will actually read
df['form_diff'] = df['w_form_10'] - df['b_form_10']
df['elo_trend_diff'] = df['w_elo_trend'] - df['b_elo_trend']

print(f"Engineered features for {len(df)} games!")
# Display the tail to verify
display(df[['date', 'white', 'black', 'form_diff', 'elo_trend_diff', 'target_result']].tail(10))

Loading data and engineering Momentum features...
Engineered features for 212241 games!


,date,white,black,form_diff,elo_trend_diff,target_result
212231,2026-07-05,"Abdusattorov,Nodirbek","Firouzja,Alireza",-0.30,-16.0,0
212232,2026-07-05,"Nakamura,Hi","Gelfand,B",0.35,11.0,0
212233,2026-07-05,"Nakamura,Hi","Madaminov,Mukhiddin",0.10,-44.0,-1
212234,2026-07-05,"Sheehan,Ethan","Nakamura,Hi",-0.45,39.0,1
212235,2026-07-05,"Nakamura,Hi","Durarbayli,Vasif",0.25,1.0,1
212236,2026-07-05,"Sevian,Samuel","Nakamura,Hi",0.10,5.0,-1
212237,2026-07-05,"Nakamura,Hi","So,W",0.00,-11.0,-1
212238,2026-07-05,"Woodward,Andy","Nakamura,Hi",-0.05,81.0,-1
212239,2026-07-05,"Nakamura,Hi","Khanin,S",0.25,-33.0,1
212240,2026-07-05,"Sindarov,Javokhir","Nakamura,Hi",0.20,0.0,0


In [2]:
import sqlite3
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, log_loss
import pickle
import os

print("--- Training Level-0 Momentum Expert V3 (Elo-Residual, Decorrelated) ---")

DB_PATH = '../data/omni_pundit.db'

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM baseline_features ORDER BY date ASC", conn)
conn.close()

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['white_key'] = df['white'].str.replace(" ", "").str.strip().str.lower()
df['black_key'] = df['black'].str.replace(" ", "").str.strip().str.lower()

# Reshape into a per-player timeline, carrying the OPPONENT's Elo too, so we
# can compute each player's Elo-EXPECTED score per game and compare it to
# what actually happened.
player_games = []
for row in df.itertuples():
    player_games.append({'game_id': row.game_id, 'date': row.date, 'player': row.white_key,
                          'color': 'white', 'own_elo': row.white_elo, 'opp_elo': row.black_elo,
                          'score': 1.0 if row.target_result == 1 else (0.5 if row.target_result == 0 else 0.0)})
    player_games.append({'game_id': row.game_id, 'date': row.date, 'player': row.black_key,
                          'color': 'black', 'own_elo': row.black_elo, 'opp_elo': row.white_elo,
                          'score': 1.0 if row.target_result == -1 else (0.5 if row.target_result == 0 else 0.0)})

pdf = pd.DataFrame(player_games).sort_values('date')

# THE CORE FIX: instead of "did they win" (mostly explained by Elo alone -
# that's already Baseline's job), compute "did they do BETTER OR WORSE than
# their Elo predicted." Standard chess expected-score formula. A residual
# near 0 means "playing exactly to rating." Positive = overperforming (in
# form), negative = underperforming (out of form) - regardless of whether
# they're rated 2400 or 2800. This is what actually decorrelates Momentum
# from Baseline/ECO, instead of just being a smoothed copy of Elo.
pdf['expected_score'] = 1 / (1 + 10 ** (-(pdf['own_elo'] - pdf['opp_elo']) / 400))
pdf['residual'] = pdf['score'] - pdf['expected_score']

# Shift by 1 so the CURRENT game's own residual never leaks into its own
# "past form" feature.
pdf['past_residual'] = pdf.groupby('player')['residual'].shift(1)
pdf['past_elo'] = pdf.groupby('player')['own_elo'].shift(1)

WINDOW = 10
pdf['rolling_residual'] = pdf.groupby('player')['past_residual'].rolling(window=WINDOW, min_periods=1).mean().reset_index(0, drop=True)
pdf['rolling_residual_vol'] = pdf.groupby('player')['past_residual'].rolling(window=WINDOW, min_periods=1).std().reset_index(0, drop=True)
pdf['rolling_residual_vol'] = pdf['rolling_residual_vol'].fillna(0)

pdf['elo_10_games_ago'] = pdf.groupby('player')['past_elo'].shift(WINDOW - 1)
pdf['elo_trend'] = pdf['own_elo'] - pdf['elo_10_games_ago']

# Color-specific residual: how does this player perform relative to Elo
# expectation SPECIFICALLY with this color.
pdf['color_past_residual'] = pdf.groupby(['player', 'color'])['residual'].shift(1)
pdf['color_residual_form'] = pdf.groupby(['player', 'color'])['color_past_residual'].rolling(window=5, min_periods=1).mean().reset_index([0, 1], drop=True)
pdf['color_residual_form'] = pdf['color_residual_form'].fillna(0.0)

w_adv = pdf[pdf['color'] == 'white'][['game_id', 'rolling_residual', 'rolling_residual_vol', 'elo_trend', 'color_residual_form']].rename(
    columns={'rolling_residual': 'w_residual', 'rolling_residual_vol': 'w_vol', 'elo_trend': 'w_elo_trend', 'color_residual_form': 'w_color_residual'}
)
b_adv = pdf[pdf['color'] == 'black'][['game_id', 'rolling_residual', 'rolling_residual_vol', 'elo_trend', 'color_residual_form']].rename(
    columns={'rolling_residual': 'b_residual', 'rolling_residual_vol': 'b_vol', 'elo_trend': 'b_elo_trend', 'color_residual_form': 'b_color_residual'}
)

assert w_adv['game_id'].is_unique, "Duplicate game_ids detected in white features!"
assert b_adv['game_id'].is_unique, "Duplicate game_ids detected in black features!"

df = df.merge(w_adv, on='game_id', how='left')
df = df.merge(b_adv, on='game_id', how='left')

df.fillna({'w_residual': 0.0, 'b_residual': 0.0, 'w_vol': 0.0, 'b_vol': 0.0,
           'w_elo_trend': 0.0, 'b_elo_trend': 0.0, 'w_color_residual': 0.0, 'b_color_residual': 0.0}, inplace=True)

df['residual_diff'] = df['w_residual'] - df['b_residual']
df['elo_trend_diff'] = df['w_elo_trend'] - df['b_elo_trend']
df['vol_diff'] = df['w_vol'] - df['b_vol']
df['color_residual_diff'] = df['w_color_residual'] - df['b_color_residual']

train_mask = df['date'].dt.year < 2026
df_train = df[train_mask].copy()
y_mapped = df_train['target_result'].map({-1: 0, 0: 1, 1: 2})

momentum_features_v3 = ['residual_diff', 'elo_trend_diff', 'vol_diff', 'color_residual_diff']
X = df_train[momentum_features_v3]

xgb_expert_v3 = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, objective='multi:softprob',
    eval_metric='mlogloss', n_jobs=-1, random_state=42
)

tscv = TimeSeriesSplit(n_splits=5)
oof_probs = np.full((len(df_train), 3), np.nan)

print(f"Generating Walk-Forward OOF Predictions across {len(df_train)} historical games...")
X_vals = X.values
y_vals = y_mapped.values

for train_index, test_index in tscv.split(X_vals):
    X_train, X_test = X_vals[train_index], X_vals[test_index]
    y_train = y_vals[train_index]
    xgb_expert_v3.fit(X_train, y_train)
    oof_probs[test_index] = xgb_expert_v3.predict_proba(X_test)

oof_preds = np.argmax(oof_probs, axis=1)
valid_idx = ~np.isnan(oof_probs[:, 0])

# XGBoost's predict_proba returns float32, which loses enough precision that
# row sums can drift a hair from exactly 1.0 - enough to trip sklearn's
# strict sum-to-one check in log_loss (LogisticRegression's float64 output
# doesn't have this issue). Cast to float64 and renormalize before scoring;
# this doesn't change the actual predictions, just their stored precision.
scored_probs = oof_probs[valid_idx].astype(np.float64)
scored_probs = scored_probs / scored_probs.sum(axis=1, keepdims=True)

acc = accuracy_score(y_vals[valid_idx], oof_preds[valid_idx])
loss = log_loss(y_vals[valid_idx], scored_probs)

print(f"\nMomentum Expert V3 OOF Accuracy: {acc * 100:.2f}%")
print(f"Momentum Expert V3 OOF Log Loss: {loss:.4f}")
print("(Standalone accuracy may be LOWER than V2 - that's expected and fine. V3's job")
print(" isn't to be individually accurate, it's to add a genuinely different opinion")
print(" the Baseline/ECO experts can't already provide. Re-run correlation_check.py")
print(" after this to confirm the redundancy actually dropped.)")

df_train['momentum_oof_prob_black'] = oof_probs[:, 0]
df_train['momentum_oof_prob_draw'] = oof_probs[:, 1]
df_train['momentum_oof_prob_white'] = oof_probs[:, 2]

oof_features = df_train[['game_id', 'momentum_oof_prob_black', 'momentum_oof_prob_draw', 'momentum_oof_prob_white']].copy()
oof_features.dropna(inplace=True)

conn = sqlite3.connect(DB_PATH)
oof_features.to_sql('oof_predictions_momentum', conn, if_exists='replace', index=False)
conn.close()
print(f"Saved {len(oof_features)} OOF predictions to SQLite 'oof_predictions_momentum' table (replaces V2).")

xgb_expert_v3.fit(X_vals, y_vals)
model_path = '../saved_models/momentum_expert_xgb.pkl'
os.makedirs(os.path.dirname(model_path), exist_ok=True)
with open(model_path, 'wb') as f:
    pickle.dump({'model': xgb_expert_v3, 'features': momentum_features_v3}, f)

print(f"Success! Model & features saved to {model_path}")

--- Training Level-0 Momentum Expert V3 (Elo-Residual, Decorrelated) ---
Generating Walk-Forward OOF Predictions across 205647 historical games...

Momentum Expert V3 OOF Accuracy: 43.96%
Momentum Expert V3 OOF Log Loss: 1.0351
(Standalone accuracy may be LOWER than V2 - that's expected and fine. V3's job
 isn't to be individually accurate, it's to add a genuinely different opinion
 the Baseline/ECO experts can't already provide. Re-run correlation_check.py
 after this to confirm the redundancy actually dropped.)
Saved 171370 OOF predictions to SQLite 'oof_predictions_momentum' table (replaces V2).
Success! Model & features saved to ../saved_models/momentum_expert_xgb.pkl


In [3]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
import scipy.sparse as sp
import pickle
import os

print("--- Training Level-0 Opening (ECO) Expert V3 (Elo-Free, Decorrelated) ---")

DB_PATH = '../data/omni_pundit.db'

conn = sqlite3.connect(DB_PATH)
query = """
    SELECT b.game_id, b.date, b.white, b.black, c.opening, b.target_result
    FROM baseline_features b
    JOIN chess_games c ON b.game_id = c.id
    ORDER BY b.date ASC
"""
df_eco = pd.read_sql(query, conn)
conn.close()

df_eco['date'] = pd.to_datetime(df_eco['date'], errors='coerce')
df_eco['opening'] = df_eco['opening'].fillna('Unknown')
df_eco['white_key'] = df_eco['white'].str.replace(" ", "").str.strip().str.lower()
df_eco['black_key'] = df_eco['black'].str.replace(" ", "").str.strip().str.lower()

# THE CORE FIX: V2 included elo_diff/has_elo, which let this expert lean on
# the same Elo signal Baseline already provides (0.965 correlation - almost
# the same model wearing a different name). Stripping Elo out entirely
# forces this expert to stand on opening-specific information alone:
# (a) the raw opening choice via one-hot encoding, and
# (b) NEW - each player's own historical track record in that SPECIFIC
#     opening, which Baseline/Momentum have no way to see at all.
player_opening_games = []
for row in df_eco.itertuples():
    score_w = 1.0 if row.target_result == 1 else (0.5 if row.target_result == 0 else 0.0)
    score_b = 1.0 if row.target_result == -1 else (0.5 if row.target_result == 0 else 0.0)
    player_opening_games.append({'game_id': row.game_id, 'date': row.date, 'player': row.white_key,
                                  'color': 'white', 'opening': row.opening, 'score': score_w})
    player_opening_games.append({'game_id': row.game_id, 'date': row.date, 'player': row.black_key,
                                  'color': 'black', 'opening': row.opening, 'score': score_b})

pog = pd.DataFrame(player_opening_games).sort_values('date')

# Expanding (all-time-so-far) mean score for this exact player+opening
# combo, shifted by 1 so the current game's own result never leaks into its
# own feature.
pog['past_score'] = pog.groupby(['player', 'opening'])['score'].shift(1)
pog['opening_familiarity'] = pog.groupby(['player', 'opening'])['past_score'].expanding().mean().reset_index([0, 1], drop=True)
# A player's first time in a given opening has no history yet - treat as
# neutral (0.5) rather than assuming they're good or bad at it.
pog['opening_familiarity'] = pog['opening_familiarity'].fillna(0.5)

w_fam = pog[pog['color'] == 'white'][['game_id', 'opening_familiarity']].rename(columns={'opening_familiarity': 'w_opening_familiarity'})
b_fam = pog[pog['color'] == 'black'][['game_id', 'opening_familiarity']].rename(columns={'opening_familiarity': 'b_opening_familiarity'})

assert w_fam['game_id'].is_unique, "Duplicate game_ids detected in white opening familiarity!"
assert b_fam['game_id'].is_unique, "Duplicate game_ids detected in black opening familiarity!"

df_eco = df_eco.merge(w_fam, on='game_id', how='left')
df_eco = df_eco.merge(b_fam, on='game_id', how='left')
df_eco.fillna({'w_opening_familiarity': 0.5, 'b_opening_familiarity': 0.5}, inplace=True)

df_eco['opening_familiarity_diff'] = df_eco['w_opening_familiarity'] - df_eco['b_opening_familiarity']

train_mask = df_eco['date'].dt.year < 2026
df_train = df_eco[train_mask].copy()
y = df_train['target_result']

print(f"Processing {len(df_train)} historical games...")

# Sparse OHE of opening + dense familiarity features - NO Elo anywhere in
# this feature set.
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_eco_sparse = encoder.fit_transform(df_train[['opening']])

scaler = StandardScaler()
X_dense = scaler.fit_transform(df_train[['w_opening_familiarity', 'b_opening_familiarity', 'opening_familiarity_diff']])

X_combined = sp.hstack((X_eco_sparse, X_dense)).tocsr()
y_vals = y.values

eco_expert_v3 = LogisticRegression(max_iter=1000, solver='lbfgs')
tscv = TimeSeriesSplit(n_splits=5)
oof_probs = np.full((len(df_train), 3), np.nan)

print("Generating Walk-Forward OOF Predictions (Opening + Player-Opening History, Elo-Free)...")
for train_index, test_index in tscv.split(X_combined):
    X_train, X_test = X_combined[train_index], X_combined[test_index]
    y_train = y_vals[train_index]
    eco_expert_v3.fit(X_train, y_train)
    oof_probs[test_index] = eco_expert_v3.predict_proba(X_test)

# LR classes are [-1, 0, 1] here (predict_proba column order follows sorted
# class order); argmax gives [0,1,2], so shift by -1 to map back.
oof_preds = np.argmax(oof_probs, axis=1) - 1
valid_idx = ~np.isnan(oof_probs[:, 0])

acc = accuracy_score(y_vals[valid_idx], oof_preds[valid_idx])
loss = log_loss(y_vals[valid_idx], oof_probs[valid_idx], labels=[-1, 0, 1])

print(f"\nECO Expert V3 OOF Accuracy: {acc * 100:.2f}%")
print(f"ECO Expert V3 OOF Log Loss: {loss:.4f}")
print("(Standalone accuracy will likely drop from V2 now that Elo is gone - expected.")
print(" This expert's job is to add opening-specific signal the others can't see,")
print(" not to individually top the leaderboard. Re-run correlation_check.py after")
print(" this to confirm the redundancy with Baseline actually dropped.)")

df_train['eco_oof_prob_black'] = oof_probs[:, 0]
df_train['eco_oof_prob_draw'] = oof_probs[:, 1]
df_train['eco_oof_prob_white'] = oof_probs[:, 2]

oof_features = df_train[['game_id', 'eco_oof_prob_black', 'eco_oof_prob_draw', 'eco_oof_prob_white']].copy()
oof_features.dropna(inplace=True)

conn = sqlite3.connect(DB_PATH)
oof_features.to_sql('oof_predictions_eco', conn, if_exists='replace', index=False)
conn.close()
print(f"Saved {len(oof_features)} OOF predictions to SQLite 'oof_predictions_eco' table (replaces V2).")

eco_expert_v3.fit(X_combined, y_vals)

model_path = '../saved_models/eco_expert_lr.pkl'
os.makedirs(os.path.dirname(model_path), exist_ok=True)
with open(model_path, 'wb') as f:
    pickle.dump({'model': eco_expert_v3, 'encoder': encoder, 'scaler': scaler}, f)

print(f"Success! ECO Expert V3 saved to {model_path}")

--- Training Level-0 Opening (ECO) Expert V3 (Elo-Free, Decorrelated) ---
Processing 205647 historical games...
Generating Walk-Forward OOF Predictions (Opening + Player-Opening History, Elo-Free)...

ECO Expert V3 OOF Accuracy: 47.39%
ECO Expert V3 OOF Log Loss: 1.0397
(Standalone accuracy will likely drop from V2 now that Elo is gone - expected.
 This expert's job is to add opening-specific signal the others can't see,
 not to individually top the leaderboard. Re-run correlation_check.py after
 this to confirm the redundancy with Baseline actually dropped.)
Saved 171370 OOF predictions to SQLite 'oof_predictions_eco' table (replaces V2).
Success! ECO Expert V3 saved to ../saved_models/eco_expert_lr.pkl


In [4]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.calibration import calibration_curve
import pickle
import os

print("--- Training Level-0 Baseline Expert V2 (Leak-Proof) ---")

# 1. Fetch data
conn = sqlite3.connect('../data/omni_pundit.db')
query = """
    SELECT game_id, date, white_elo, black_elo, elo_diff, has_elo,
           white_historical_wins, black_historical_wins, historical_draws, 
           target_result 
    FROM baseline_features
    ORDER BY date ASC
"""
df_base = pd.read_sql(query, conn)
conn.close()

df_base['date'] = pd.to_datetime(df_base['date'], errors='coerce')

# 2. Vault the 2026 Acid Test Data
train_mask = df_base['date'].dt.year < 2026
df_train = df_base[train_mask].copy()

# 3. Define Features
feature_cols = ['white_elo', 'black_elo', 'elo_diff', 'has_elo',
                'white_historical_wins', 'black_historical_wins', 'historical_draws']

X = df_train[feature_cols].values
y = df_train['target_result'].values

# 4. Strict Walk-Forward Validation
tscv = TimeSeriesSplit(n_splits=5)
baseline_expert = LogisticRegression(max_iter=1000)

oof_probs = np.full((len(df_train), 3), np.nan)

print(f"Generating Walk-Forward OOF Predictions across {len(df_train)} historical games...")

for train_index, test_index in tscv.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train = y[train_index]
    
    # CRITICAL: We fit the scaler ONLY on the past data. 
    # This prevents the model from knowing the mean/variance of future Elo ratings!
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    baseline_expert.fit(X_train_scaled, y_train)
    oof_probs[test_index] = baseline_expert.predict_proba(X_test_scaled)

# 5. Evaluate (Only on valid predicted folds)
# LR classes are [-1, 0, 1]. np.argmax gives [0, 1, 2], so we shift by -1.
oof_preds = np.argmax(oof_probs, axis=1) - 1
valid_idx = ~np.isnan(oof_probs[:, 0])

acc = accuracy_score(y[valid_idx], oof_preds[valid_idx])
loss = log_loss(y[valid_idx], oof_probs[valid_idx], labels=[-1, 0, 1])

print(f"\nBaseline Expert V2 OOF Accuracy: {acc * 100:.2f}%")
print(f"Baseline Expert V2 OOF Log Loss: {loss:.4f}")

# 6. Walk-Forward Calibration Check
# We check calibration for class '1' (White Win) to ensure the probabilities are grounded
most_confident_class_idx = np.where(baseline_expert.classes_ == 1)[0][0]
class_probs = oof_probs[valid_idx, most_confident_class_idx]
y_val_binary = [1 if val == 1 else 0 for val in y[valid_idx]]

try:
    prob_true, prob_pred = calibration_curve(y_val_binary, class_probs, n_bins=5, strategy='quantile')
    print(f"\nCalibration check for White Wins (predicted vs actual):")
    for pt, pp in zip(prob_true, prob_pred):
        print(f"  Predicted ~{pp*100:.1f}% -> Actually happened {pt*100:.1f}% of the time")
except Exception as e:
    print(f"\nCalibration curve failed: {e}")

# 7. Save OOF to SQLite for the Meta-Learner
df_train['base_oof_prob_black'] = oof_probs[:, 0]
df_train['base_oof_prob_draw'] = oof_probs[:, 1]
df_train['base_oof_prob_white'] = oof_probs[:, 2]

oof_features = df_train[['game_id', 'base_oof_prob_black', 'base_oof_prob_draw', 'base_oof_prob_white']].copy()
oof_features.dropna(inplace=True)

conn = sqlite3.connect('../data/omni_pundit.db')
oof_features.to_sql('oof_predictions_baseline', conn, if_exists='replace', index=False)
conn.close()
print(f"\nSaved {len(oof_features)} OOF predictions to SQLite 'oof_predictions_baseline' table.")

# 8. Train Final & Pickle (Must fit a final scaler on ALL pre-2026 data for future inference)
final_scaler = StandardScaler()
X_final_scaled = final_scaler.fit_transform(X)
baseline_expert.fit(X_final_scaled, y)

model_path = '../saved_models/baseline_expert.pkl'
os.makedirs(os.path.dirname(model_path), exist_ok=True)
with open(model_path, 'wb') as f:
    pickle.dump({'model': baseline_expert, 'scaler': final_scaler}, f)
    
print(f"Success! Model and Scaler saved to {model_path}")

--- Training Level-0 Baseline Expert V2 (Leak-Proof) ---
Generating Walk-Forward OOF Predictions across 205647 historical games...

Baseline Expert V2 OOF Accuracy: 59.18%
Baseline Expert V2 OOF Log Loss: 0.9158

Calibration check for White Wins (predicted vs actual):
  Predicted ~5.8% -> Actually happened 12.9% of the time
  Predicted ~17.9% -> Actually happened 21.5% of the time
  Predicted ~32.2% -> Actually happened 32.1% of the time
  Predicted ~52.6% -> Actually happened 53.6% of the time
  Predicted ~78.8% -> Actually happened 79.3% of the time

Saved 171370 OOF predictions to SQLite 'oof_predictions_baseline' table.
Success! Model and Scaler saved to ../saved_models/baseline_expert.pkl


In [5]:
import sqlite3
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, log_loss
import pickle
import os

print("--- Training Level-0 ACPL (Engine Accuracy) Expert (Leak-Proof) ---")

# 1. Fetch the ACPL data from the database
conn = sqlite3.connect('../data/omni_pundit.db')
query = """
    SELECT b.game_id, b.date, b.white, b.black, b.target_result, 
           c.white_acpl, c.black_acpl
    FROM baseline_features b
    JOIN chess_games c ON b.game_id = c.id
    WHERE c.white_acpl IS NOT NULL
    ORDER BY b.date ASC
"""
df_acpl = pd.read_sql(query, conn)
conn.close()

df_acpl['date'] = pd.to_datetime(df_acpl['date'], errors='coerce')
print(f"Loaded {len(df_acpl)} games with Stockfish evaluation data.")

# 2. Reshape into timelines to calculate Rolling ACPL safely
df_acpl['white_key'] = df_acpl['white'].str.replace(" ", "").str.strip().str.lower()
df_acpl['black_key'] = df_acpl['black'].str.replace(" ", "").str.strip().str.lower()

player_acpl = []
for row in df_acpl.itertuples():
    player_acpl.append({'game_id': row.game_id, 'date': row.date, 'player': row.white_key, 'color': 'white', 'acpl': row.white_acpl})
    player_acpl.append({'game_id': row.game_id, 'date': row.date, 'player': row.black_key, 'color': 'black', 'acpl': row.black_acpl})

pdf_acpl = pd.DataFrame(player_acpl).sort_values('date')

# Calculate Rolling ACPL over the last 5 games (Shifted by 1 to prevent leakage!)
pdf_acpl['past_acpl'] = pdf_acpl.groupby('player')['acpl'].shift(1)
pdf_acpl['rolling_acpl'] = pdf_acpl.groupby('player')['past_acpl'].rolling(window=5, min_periods=1).mean().reset_index(0, drop=True)

# Re-merge the rolling features safely
w_acpl_df = pdf_acpl[pdf_acpl['color'] == 'white'][['game_id', 'rolling_acpl']].rename(columns={'rolling_acpl': 'w_rolling_acpl'})
b_acpl_df = pdf_acpl[pdf_acpl['color'] == 'black'][['game_id', 'rolling_acpl']].rename(columns={'rolling_acpl': 'b_rolling_acpl'})

assert w_acpl_df['game_id'].is_unique, "Duplicate game_ids detected in white ACPL features!"
assert b_acpl_df['game_id'].is_unique, "Duplicate game_ids detected in black ACPL features!"

df_acpl = df_acpl.merge(w_acpl_df, on='game_id', how='left')
df_acpl = df_acpl.merge(b_acpl_df, on='game_id', how='left')

# Drop rows where we don't have historical ACPL yet
df_acpl = df_acpl.dropna(subset=['w_rolling_acpl', 'b_rolling_acpl']).copy()

# Feature: White's Rolling ACPL minus Black's Rolling ACPL
df_acpl['acpl_diff'] = df_acpl['w_rolling_acpl'] - df_acpl['b_rolling_acpl']

# 3. Vault the 2026 Data
train_mask = df_acpl['date'].dt.year < 2026
df_train = df_acpl[train_mask].copy()

acpl_features = ['acpl_diff', 'w_rolling_acpl', 'b_rolling_acpl']
X = df_train[acpl_features].values
y_mapped = df_train['target_result'].map({-1: 0, 0: 1, 1: 2}).values

print(f"Training on {len(df_train)} games with historical engine context...")

# 4. Train the ACPL Expert (XGBoost)
acpl_expert = XGBClassifier(
    n_estimators=150, max_depth=3, learning_rate=0.05,
    objective='multi:softprob', eval_metric='mlogloss', n_jobs=-1, random_state=42
)

# Strict Walk-Forward Validation
tscv = TimeSeriesSplit(n_splits=5)
oof_probs = np.full((len(df_train), 3), np.nan)

print("Generating Walk-Forward ACPL OOF Predictions...")
for train_index, test_index in tscv.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train = y_mapped[train_index]
    
    acpl_expert.fit(X_train, y_train)
    oof_probs[test_index] = acpl_expert.predict_proba(X_test)

# 5. Evaluate (Only on valid predicted folds)
oof_preds = np.argmax(oof_probs, axis=1)
valid_idx = ~np.isnan(oof_probs[:, 0])

acc = accuracy_score(y_mapped[valid_idx], oof_preds[valid_idx])
loss = log_loss(y_mapped[valid_idx], oof_probs[valid_idx])

print(f"\nACPL Expert V2 OOF Accuracy: {acc * 100:.2f}%")
print(f"ACPL Expert V2 OOF Log Loss: {loss:.4f}")

# 6. Save OOF to SQLite for the Meta-Learner
df_train['acpl_oof_prob_black'] = oof_probs[:, 0]
df_train['acpl_oof_prob_draw'] = oof_probs[:, 1]
df_train['acpl_oof_prob_white'] = oof_probs[:, 2]

oof_features = df_train[['game_id', 'acpl_oof_prob_black', 'acpl_oof_prob_draw', 'acpl_oof_prob_white']].copy()
oof_features.dropna(inplace=True)

conn = sqlite3.connect('../data/omni_pundit.db')
oof_features.to_sql('oof_predictions_acpl', conn, if_exists='replace', index=False)
conn.close()
print(f"Saved {len(oof_features)} ACPL OOF predictions to SQLite 'oof_predictions_acpl' table.")

# 7. Train Final & Pickle
acpl_expert.fit(X, y_mapped)
model_path = '../saved_models/acpl_expert_xgb.pkl'
os.makedirs(os.path.dirname(model_path), exist_ok=True)

with open(model_path, 'wb') as f:
    pickle.dump({'model': acpl_expert, 'features': acpl_features}, f)

print(f"Success! ACPL Expert saved to {model_path}")

--- Training Level-0 ACPL (Engine Accuracy) Expert (Leak-Proof) ---
Loaded 9994 games with Stockfish evaluation data.
Training on 6425 games with historical engine context...
Generating Walk-Forward ACPL OOF Predictions...

ACPL Expert V2 OOF Accuracy: 42.36%
ACPL Expert V2 OOF Log Loss: 1.0741
Saved 5350 ACPL OOF predictions to SQLite 'oof_predictions_acpl' table.
Success! ACPL Expert saved to ../saved_models/acpl_expert_xgb.pkl


c:\Users\harsh\anaconda3\envs\crime_scene_env\lib\site-packages\sklearn\metrics\_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


In [6]:
import sqlite3
import pandas as pd

DB_PATH = '../data/omni_pundit.db'

def check_expert_correlation():
    print("--- Checking correlation between base expert OOF predictions ---")
    conn = sqlite3.connect(DB_PATH)
    query = """
        SELECT
            base.base_oof_prob_white, base.base_oof_prob_draw, base.base_oof_prob_black,
            eco.eco_oof_prob_white, eco.eco_oof_prob_draw, eco.eco_oof_prob_black,
            mom.momentum_oof_prob_white, mom.momentum_oof_prob_draw, mom.momentum_oof_prob_black
        FROM oof_predictions_baseline base
        JOIN oof_predictions_eco eco ON base.game_id = eco.game_id
        JOIN oof_predictions_momentum mom ON base.game_id = mom.game_id
    """
    df = pd.read_sql(query, conn)
    conn.close()

    print(f"Rows compared: {len(df)}\n")

    # Focus on P(White Win) from each expert - if two experts are giving
    # near-identical opinions on the same games, the meta-learner isn't
    # getting 9 independent signals, it's getting fewer, no matter how big
    # the network is. This tells you whether the ceiling right now is
    # architecture-bound or information-bound.
    white_prob_cols = ['base_oof_prob_white', 'eco_oof_prob_white', 'momentum_oof_prob_white']
    corr_white = df[white_prob_cols].corr()
    print("Correlation matrix - P(White Win) across experts:")
    print(corr_white.round(3))
    print()

    full_corr = df.corr()
    print("Full 9x9 correlation matrix (all classes, all experts):")
    print(full_corr.round(2))
    print()

    # Flag any cross-expert pair above 0.85 - a rough "these two are saying
    # almost the same thing" threshold. We deliberately skip same-expert
    # pairs (e.g. base_white vs base_black) since those are trivially
    # anti-correlated by construction - they're compositional (roughly sum
    # to 1 with draw), so a strong correlation there is meaningless noise,
    # not a sign of redundant experts.
    print("Cross-expert pairs with |correlation| > 0.85 (excluding same-expert pairs):")
    flagged = False
    cols = full_corr.columns

    def expert_of(col):
        return col.split('_oof_prob_')[0]

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            if expert_of(cols[i]) == expert_of(cols[j]):
                continue  # same expert, skip - see comment above
            val = full_corr.iloc[i, j]
            if abs(val) > 0.85:
                print(f"  {cols[i]}  <->  {cols[j]}:  {val:.3f}")
                flagged = True
    if not flagged:
        print("  None found - experts appear reasonably decorrelated.")

if __name__ == "__main__":
    check_expert_correlation()

--- Checking correlation between base expert OOF predictions ---
Rows compared: 171370

Correlation matrix - P(White Win) across experts:
                         base_oof_prob_white  eco_oof_prob_white  \
base_oof_prob_white                    1.000               0.463   
eco_oof_prob_white                     0.463               1.000   
momentum_oof_prob_white                0.545               0.202   

                         momentum_oof_prob_white  
base_oof_prob_white                        0.545  
eco_oof_prob_white                         0.202  
momentum_oof_prob_white                    1.000  

Full 9x9 correlation matrix (all classes, all experts):
                         base_oof_prob_white  base_oof_prob_draw  \
base_oof_prob_white                     1.00               -0.46   
base_oof_prob_draw                     -0.46                1.00   
base_oof_prob_black                    -0.82               -0.13   
eco_oof_prob_white                      0.46            

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import TimeSeriesSplit
import pickle
import os

print("--- Training Meta-Learner V2 (Wide & Deep + Psych Context) ---")

DB_PATH = '../data/omni_pundit.db'
MODEL_PATH = '../saved_models/meta_learner_wide_deep.pkl'

# ============================================================
# 1. Load & engineer features (unchanged from your Experiment #04)
# ============================================================
conn = sqlite3.connect(DB_PATH)
query = """
    SELECT
        b.game_id, b.date, b.white, b.black, b.target_result, b.elo_diff,
        base.base_oof_prob_black, base.base_oof_prob_draw, base.base_oof_prob_white,
        eco.eco_oof_prob_black, eco.eco_oof_prob_draw, eco.eco_oof_prob_white,
        mom.momentum_oof_prob_black, mom.momentum_oof_prob_draw, mom.momentum_oof_prob_white
    FROM oof_predictions_baseline base
    JOIN oof_predictions_eco eco ON base.game_id = eco.game_id
    JOIN oof_predictions_momentum mom ON base.game_id = mom.game_id
    JOIN baseline_features b ON base.game_id = b.game_id
    ORDER BY b.date ASC
"""
df_meta = pd.read_sql(query, conn)
conn.close()

df_meta['date'] = pd.to_datetime(df_meta['date'])

print("Engineering Psychological Context (Tilt & Fatigue)...")
player_timeline = []
for row in df_meta.itertuples():
    player_timeline.append({'game_id': row.game_id, 'date': row.date, 'player': row.white,
                             'color': 'white',
                             'result_class': 1 if row.target_result == 1 else (0 if row.target_result == 0 else -1)})
    player_timeline.append({'game_id': row.game_id, 'date': row.date, 'player': row.black,
                             'color': 'black',
                             'result_class': 1 if row.target_result == -1 else (0 if row.target_result == 0 else -1)})

pdf = pd.DataFrame(player_timeline).sort_values(['player', 'date']).copy()

pdf['prev_date'] = pdf.groupby('player')['date'].shift(1)
pdf['rest_days'] = (pdf['date'] - pdf['prev_date']).dt.days.fillna(30)
pdf['rest_days'] = np.clip(pdf['rest_days'], 0, 90)

pdf = pdf.set_index('date')
fatigue_values = pdf.groupby('player')['game_id'].rolling('7D').count().values - 1
pdf['fatigue_7d'] = fatigue_values
pdf = pdf.reset_index()

# FIX: tilt should only accumulate on genuine losses (result_class == -1),
# not draws. Draws at GM level - especially with Black - are often a
# deliberate strategic choice, not a symptom of "tilt". Lumping them in
# with losses diluted the signal this feature was meant to capture.
pdf['loss_flag'] = (pdf['result_class'] == -1).astype(int)
pdf['tilt_streak'] = pdf.groupby('player')['loss_flag'].transform(
    lambda x: x * (x.groupby((x != x.shift()).cumsum()).cumcount() + 1)
)
pdf['tilt_streak'] = pdf.groupby('player')['tilt_streak'].shift(1).fillna(0)

# NEW: separate feature for draw tendency specifically with Black - a
# distinct signal from tilt, since a player drawing on purpose isn't in
# the same psychological state as a player who's been losing.
pdf['draw_flag'] = (pdf['result_class'] == 0).astype(int)
pdf['black_draw_flag'] = np.where(pdf['color'] == 'black', pdf['draw_flag'], np.nan)
pdf['black_draw_rate_10'] = pdf.groupby('player')['black_draw_flag'].transform(
    lambda x: x.shift(1).rolling(window=10, min_periods=1).mean()
)
pdf['black_draw_rate_10'] = pdf['black_draw_rate_10'].fillna(0.0)

pdf = pdf.sort_values('date')

w_context = pdf[pdf['color'] == 'white'][['game_id', 'rest_days', 'fatigue_7d', 'tilt_streak']].rename(
    columns={'rest_days': 'w_rest', 'fatigue_7d': 'w_fatigue', 'tilt_streak': 'w_tilt'})
b_context = pdf[pdf['color'] == 'black'][['game_id', 'rest_days', 'fatigue_7d', 'tilt_streak', 'black_draw_rate_10']].rename(
    columns={'rest_days': 'b_rest', 'fatigue_7d': 'b_fatigue', 'tilt_streak': 'b_tilt'})

df_meta = df_meta.merge(w_context, on='game_id', how='left')
df_meta = df_meta.merge(b_context, on='game_id', how='left')

df_meta['rest_diff'] = df_meta['w_rest'] - df_meta['b_rest']
df_meta['fatigue_diff'] = df_meta['w_fatigue'] - df_meta['b_fatigue']
df_meta['tilt_diff'] = df_meta['w_tilt'] - df_meta['b_tilt']
# black_draw_rate_10 is Black-specific (asymmetric by design - White doesn't
# have a symmetrical "plays for draws" convention the same way), kept as its
# own feature rather than forced into a diff.

df_meta.dropna(inplace=True)
df_meta = df_meta.sort_values('date').reset_index(drop=True)

wide_cols = ['elo_diff', 'rest_diff', 'fatigue_diff', 'tilt_diff', 'black_draw_rate_10']
deep_cols = [c for c in df_meta.columns if 'oof_prob' in c]

print(f"Wide features ({len(wide_cols)}): {wide_cols}")
print(f"Deep features ({len(deep_cols)}): {deep_cols}")

X_wide_all = df_meta[wide_cols].values
X_deep_all = df_meta[deep_cols].values
y_all = df_meta['target_result'].map({-1: 0, 0: 1, 1: 2}).values

# ============================================================
# 2. Model definition
# ============================================================
class WideAndDeepContext(nn.Module):
    def __init__(self, deep_in, wide_in, hidden_size=24, combined_hidden=16):
        super().__init__()
        self.deep_path = nn.Sequential(
            nn.Linear(deep_in, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        # FIX: previously the concatenated (deep_out + wide) vector went
        # straight into a single Linear -> that's a purely linear
        # combination, meaning the network could never learn interactions
        # like "when tilt is high, weight the Momentum expert more."
        # Adding one more hidden layer after the concat gives it room to
        # actually learn those cross-terms, which is the whole point of
        # Wide & Deep.
        self.combine = nn.Sequential(
            nn.Linear(hidden_size + wide_in, combined_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(combined_hidden, 3)
        )

    def forward(self, wide_x, deep_x):
        deep_out = self.deep_path(deep_x)
        combined = torch.cat((deep_out, wide_x), dim=1)
        return self.combine(combined)


def train_one_fold(X_wide_train, X_deep_train, y_train, X_wide_val, X_deep_val, y_val,
                    max_epochs=200, patience=15, batch_size=512):
    scaler = StandardScaler()
    X_wide_train_scaled = scaler.fit_transform(X_wide_train)
    X_wide_val_scaled = scaler.transform(X_wide_val)

    t_wide_train = torch.FloatTensor(X_wide_train_scaled.copy())
    t_deep_train = torch.FloatTensor(X_deep_train.copy())
    t_y_train = torch.LongTensor(y_train.copy())

    t_wide_val = torch.FloatTensor(X_wide_val_scaled.copy())
    t_deep_val = torch.FloatTensor(X_deep_val.copy())
    t_y_val = torch.LongTensor(y_val.copy())

    # FIX: previously one gradient step per epoch over the ENTIRE training
    # set (full-batch GD). Mini-batches with shuffling introduce useful
    # gradient noise that generally helps generalization at this data size,
    # and is the standard approach for a reason.
    train_ds = TensorDataset(t_wide_train, t_deep_train, t_y_train)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = WideAndDeepContext(deep_in=t_deep_train.shape[1], wide_in=t_wide_train.shape[1])
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    # FIX: previously trained a fixed 200 epochs with zero validation
    # tracking during training - no way to know if it overfit at epoch 60
    # and got worse by 200. Now we track val loss every epoch, keep the
    # best-performing weights, and stop early if it stops improving.
    best_val_loss = float('inf')
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        for wide_batch, deep_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(wide_batch, deep_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_outputs = model(t_wide_val, t_deep_val)
            val_loss = criterion(val_outputs, t_y_val).item()

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_outputs = model(t_wide_val, t_deep_val)
        val_probs = torch.softmax(val_outputs, dim=1).numpy()
        _, predicted = torch.max(val_outputs, 1)
        acc = (predicted.numpy() == y_val).mean()

    return model, scaler, acc, val_probs, best_val_loss, epoch + 1


# ============================================================
# 3. Walk-forward validation (previously a single static 80/20 split -
# every other layer in the pipeline uses TimeSeriesSplit, this brings the
# meta-learner in line so results are comparable and not a lucky/unlucky
# single slice)
# ============================================================
tscv = TimeSeriesSplit(n_splits=4)  # cheap model - could go higher, 4 is a reasonable start
fold_accs = []
fold_log_losses = []
fold_epochs = []

# Accumulates OOF predictions across folds, aligned to df_meta's row order,
# so they can be persisted to SQLite afterward (same pattern as the base
# experts' oof_predictions_* tables) and picked up by the Prediction Ledger
# backfill. Rows in the first fold's training window never get an OOF
# prediction (nothing exists before them to train on) - left as NaN, same
# convention as the base experts.
oof_probs_all = np.full((len(X_wide_all), 3), np.nan)

print("\nRunning walk-forward validation...")
for fold_num, (train_idx, val_idx) in enumerate(tscv.split(X_wide_all), start=1):
    model, scaler, acc, val_probs, val_loss, epochs_run = train_one_fold(
        X_wide_all[train_idx], X_deep_all[train_idx], y_all[train_idx],
        X_wide_all[val_idx], X_deep_all[val_idx], y_all[val_idx]
    )
    ll = log_loss(y_all[val_idx], val_probs, labels=[0, 1, 2])
    fold_accs.append(acc)
    fold_log_losses.append(ll)
    fold_epochs.append(epochs_run)
    oof_probs_all[val_idx] = val_probs
    print(f"  Fold {fold_num}: acc={acc*100:.2f}%  log_loss={ll:.4f}  "
          f"(stopped after {epochs_run} epochs, val set size={len(val_idx)})")

print(f"\nWalk-forward mean accuracy:  {np.mean(fold_accs)*100:.2f}%  (std: {np.std(fold_accs)*100:.2f}%)")
print(f"Walk-forward mean log loss:  {np.mean(fold_log_losses):.4f}  (std: {np.std(fold_log_losses):.4f})")
print("(The std tells you how much this number would've swung on a different single split -")
print(" a wide std means the old single-split 63.07% could easily have been a lucky draw.)")

# Persist OOF predictions to SQLite - same convention as
# oof_predictions_baseline/eco/momentum, so backfill_ledger.py can pick this
# layer up too, completing the ledger across all 4 layers instead of just
# the 3 base experts.
df_meta['meta_oof_prob_black'] = oof_probs_all[:, 0]
df_meta['meta_oof_prob_draw'] = oof_probs_all[:, 1]
df_meta['meta_oof_prob_white'] = oof_probs_all[:, 2]

oof_export = df_meta[['game_id', 'meta_oof_prob_black', 'meta_oof_prob_draw', 'meta_oof_prob_white']].dropna()
conn = sqlite3.connect(DB_PATH)
oof_export.to_sql('oof_predictions_meta', conn, if_exists='replace', index=False)
conn.close()
print(f"Saved {len(oof_export)} meta-learner OOF predictions to SQLite 'oof_predictions_meta' table.")

# ============================================================
# 4. Train final model on ALL available data, held out last fold's
#    proportion as final validation for calibration reporting
# ============================================================
final_split = int(len(X_wide_all) * 0.85)
final_model, final_scaler, final_acc, final_probs, final_val_loss, _ = train_one_fold(
    X_wide_all[:final_split], X_deep_all[:final_split], y_all[:final_split],
    X_wide_all[final_split:], X_deep_all[final_split:], y_all[final_split:]
)
print(f"\nFinal model (trained on 85%, held out last 15%): "
      f"acc={final_acc*100:.2f}%  val_log_loss={final_val_loss:.4f}")

os.makedirs('../saved_models', exist_ok=True)
with open(MODEL_PATH, 'wb') as f:
    pickle.dump({
        'model_state_dict': final_model.state_dict(),
        'wide_scaler': final_scaler,
        'wide_cols': wide_cols,
        'deep_cols': deep_cols,
        'deep_in': len(deep_cols),
        'wide_in': len(wide_cols),
    }, f)

print(f"Success! Meta-learner saved to {MODEL_PATH}")

--- Training Meta-Learner V2 (Wide & Deep + Psych Context) ---
Engineering Psychological Context (Tilt & Fatigue)...
Wide features (5): ['elo_diff', 'rest_diff', 'fatigue_diff', 'tilt_diff', 'black_draw_rate_10']
Deep features (9): ['base_oof_prob_black', 'base_oof_prob_draw', 'base_oof_prob_white', 'eco_oof_prob_black', 'eco_oof_prob_draw', 'eco_oof_prob_white', 'momentum_oof_prob_black', 'momentum_oof_prob_draw', 'momentum_oof_prob_white']

Running walk-forward validation...
  Fold 1: acc=59.24%  log_loss=0.9176  (stopped after 55 epochs, val set size=34274)
  Fold 2: acc=59.59%  log_loss=0.8963  (stopped after 42 epochs, val set size=34274)
  Fold 3: acc=62.72%  log_loss=0.8553  (stopped after 59 epochs, val set size=34274)
  Fold 4: acc=63.44%  log_loss=0.8414  (stopped after 46 epochs, val set size=34274)

Walk-forward mean accuracy:  61.25%  (std: 1.86%)
Walk-forward mean log loss:  0.8776  (std: 0.0306)
(The std tells you how much this number would've swung on a different single

In [2]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, log_loss
import pickle
import os

print("--- Meta-Learner Sanity Check: Plain Logistic Regression ---")
print("(Same 14 inputs, same walk-forward protocol as the NN - if this performs")
print(" about as well as the Wide & Deep network, the ceiling is information-bound,")
print(" not architecture-bound, and further NN tuning has low expected payoff.)\n")

DB_PATH = '../data/omni_pundit.db'

# ============================================================
# 1. Load & engineer features - IDENTICAL to meta_learner_v2.py, so the
# comparison is apples-to-apples.
# ============================================================
conn = sqlite3.connect(DB_PATH)
query = """
    SELECT
        b.game_id, b.date, b.white, b.black, b.target_result, b.elo_diff,
        base.base_oof_prob_black, base.base_oof_prob_draw, base.base_oof_prob_white,
        eco.eco_oof_prob_black, eco.eco_oof_prob_draw, eco.eco_oof_prob_white,
        mom.momentum_oof_prob_black, mom.momentum_oof_prob_draw, mom.momentum_oof_prob_white
    FROM oof_predictions_baseline base
    JOIN oof_predictions_eco eco ON base.game_id = eco.game_id
    JOIN oof_predictions_momentum mom ON base.game_id = mom.game_id
    JOIN baseline_features b ON base.game_id = b.game_id
    ORDER BY b.date ASC
"""
df_meta = pd.read_sql(query, conn)
conn.close()

df_meta['date'] = pd.to_datetime(df_meta['date'])

player_timeline = []
for row in df_meta.itertuples():
    player_timeline.append({'game_id': row.game_id, 'date': row.date, 'player': row.white,
                             'color': 'white',
                             'result_class': 1 if row.target_result == 1 else (0 if row.target_result == 0 else -1)})
    player_timeline.append({'game_id': row.game_id, 'date': row.date, 'player': row.black,
                             'color': 'black',
                             'result_class': 1 if row.target_result == -1 else (0 if row.target_result == 0 else -1)})

pdf = pd.DataFrame(player_timeline).sort_values(['player', 'date']).copy()

pdf['prev_date'] = pdf.groupby('player')['date'].shift(1)
pdf['rest_days'] = (pdf['date'] - pdf['prev_date']).dt.days.fillna(30)
pdf['rest_days'] = np.clip(pdf['rest_days'], 0, 90)

pdf = pdf.set_index('date')
fatigue_values = pdf.groupby('player')['game_id'].rolling('7D').count().values - 1
pdf['fatigue_7d'] = fatigue_values
pdf = pdf.reset_index()

pdf['loss_flag'] = (pdf['result_class'] == -1).astype(int)
pdf['tilt_streak'] = pdf.groupby('player')['loss_flag'].transform(
    lambda x: x * (x.groupby((x != x.shift()).cumsum()).cumcount() + 1)
)
pdf['tilt_streak'] = pdf.groupby('player')['tilt_streak'].shift(1).fillna(0)

pdf['draw_flag'] = (pdf['result_class'] == 0).astype(int)
pdf['black_draw_flag'] = np.where(pdf['color'] == 'black', pdf['draw_flag'], np.nan)
pdf['black_draw_rate_10'] = pdf.groupby('player')['black_draw_flag'].transform(
    lambda x: x.shift(1).rolling(window=10, min_periods=1).mean()
)
pdf['black_draw_rate_10'] = pdf['black_draw_rate_10'].fillna(0.0)

pdf = pdf.sort_values('date')

w_context = pdf[pdf['color'] == 'white'][['game_id', 'rest_days', 'fatigue_7d', 'tilt_streak']].rename(
    columns={'rest_days': 'w_rest', 'fatigue_7d': 'w_fatigue', 'tilt_streak': 'w_tilt'})
b_context = pdf[pdf['color'] == 'black'][['game_id', 'rest_days', 'fatigue_7d', 'tilt_streak', 'black_draw_rate_10']].rename(
    columns={'rest_days': 'b_rest', 'fatigue_7d': 'b_fatigue', 'tilt_streak': 'b_tilt'})

df_meta = df_meta.merge(w_context, on='game_id', how='left')
df_meta = df_meta.merge(b_context, on='game_id', how='left')

df_meta['rest_diff'] = df_meta['w_rest'] - df_meta['b_rest']
df_meta['fatigue_diff'] = df_meta['w_fatigue'] - df_meta['b_fatigue']
df_meta['tilt_diff'] = df_meta['w_tilt'] - df_meta['b_tilt']

df_meta.dropna(inplace=True)
df_meta = df_meta.sort_values('date').reset_index(drop=True)

wide_cols = ['elo_diff', 'rest_diff', 'fatigue_diff', 'tilt_diff', 'black_draw_rate_10']
deep_cols = [c for c in df_meta.columns if 'oof_prob' in c]
all_cols = wide_cols + deep_cols

print(f"Total inputs ({len(all_cols)}): {all_cols}\n")

X_all = df_meta[all_cols].values
y_all = df_meta['target_result'].map({-1: 0, 0: 1, 1: 2}).values

# ============================================================
# 2. Walk-forward validation - same n_splits as the NN for a fair comparison
# ============================================================
tscv = TimeSeriesSplit(n_splits=4)
fold_accs = []
fold_log_losses = []

print("Running walk-forward validation...")
for fold_num, (train_idx, val_idx) in enumerate(tscv.split(X_all), start=1):
    X_train, X_val = X_all[train_idx], X_all[val_idx]
    y_train, y_val = y_all[train_idx], y_all[val_idx]

    # Fit scaler on training fold only - same leakage discipline as
    # Baseline Expert V2.
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_scaled, y_train)

    probs = model.predict_proba(X_val_scaled)
    preds = model.predict(X_val_scaled)

    acc = accuracy_score(y_val, preds)
    ll = log_loss(y_val, probs, labels=model.classes_)
    fold_accs.append(acc)
    fold_log_losses.append(ll)
    print(f"  Fold {fold_num}: acc={acc*100:.2f}%  log_loss={ll:.4f}  (val set size={len(val_idx)})")

print(f"\nWalk-forward mean accuracy:  {np.mean(fold_accs)*100:.2f}%  (std: {np.std(fold_accs)*100:.2f}%)")
print(f"Walk-forward mean log loss:  {np.mean(fold_log_losses):.4f}  (std: {np.std(fold_log_losses):.4f})")
print("\nCompare directly against the NN's walk-forward numbers:")
print("  - If close: the NN's extra complexity isn't earning its keep yet -")
print("    either give it more capacity/less regularization, or accept the")
print("    linear fusion as good enough and move on to new data (e.g. ACPL).")
print("  - If the NN clearly wins: its nonlinear interactions are doing real")
print("    work, worth investing in a bigger version.")

# ============================================================
# 3. Train final model on all data for reference
# ============================================================
final_scaler = StandardScaler()
X_all_scaled = final_scaler.fit_transform(X_all)
final_model = LogisticRegression(max_iter=1000)
final_model.fit(X_all_scaled, y_all)

os.makedirs('../saved_models', exist_ok=True)
model_path = '../saved_models/meta_learner_logreg_sanity_check.pkl'
with open(model_path, 'wb') as f:
    pickle.dump({'model': final_model, 'scaler': final_scaler, 'features': all_cols}, f)

print(f"\nSaved reference model to {model_path}")

--- Meta-Learner Sanity Check: Plain Logistic Regression ---
(Same 14 inputs, same walk-forward protocol as the NN - if this performs
 about as well as the Wide & Deep network, the ceiling is information-bound,
 not architecture-bound, and further NN tuning has low expected payoff.)

Total inputs (14): ['elo_diff', 'rest_diff', 'fatigue_diff', 'tilt_diff', 'black_draw_rate_10', 'base_oof_prob_black', 'base_oof_prob_draw', 'base_oof_prob_white', 'eco_oof_prob_black', 'eco_oof_prob_draw', 'eco_oof_prob_white', 'momentum_oof_prob_black', 'momentum_oof_prob_draw', 'momentum_oof_prob_white']

Running walk-forward validation...
  Fold 1: acc=58.38%  log_loss=0.9262  (val set size=34274)
  Fold 2: acc=59.19%  log_loss=0.9003  (val set size=34274)
  Fold 3: acc=62.13%  log_loss=0.8598  (val set size=34274)
  Fold 4: acc=63.24%  log_loss=0.8458  (val set size=34274)

Walk-forward mean accuracy:  60.74%  (std: 2.01%)
Walk-forward mean log loss:  0.8830  (std: 0.0320)

Compare directly against th